# 1. Dependencies

## System and OS

In [ ]:
import os
import sys

## Paralel

In [ ]:
from joblib import Parallel, delayed, parallel_config
from joblib import parallel_config

## Data Processor

In [1]:
import numpy as np
import pandas as pd
import polars as pl

### Utilities

In [ ]:
def get_all_file_names_in_folder(folder:str,file_extension:str):
    files = []
    for file in os.listdir(folder):
        if len(folder) > 0:
            if file.endswith(f".{file_extension}"):
                files.append(file)
    return sorted(files)


# 2. Reformatting

## 2.1. Change CSV to Parquet

The raw CSE-CIC-IDS2018 dataset ships as 10 daily CSV files (~6.7 GB). CSV is slow to read, stores
every value as text and carries no schema, so the first pre-processing step rewrites it as Parquet:
columnar, compressed, typed, and readable one piece at a time.

**In:** `cse-cic-ids2018/*.csv` $\rightarrow$ **Out:** `data-raw/*.parquet` (one file per chunk of rows)

Three things have to be repaired on the way, because the dataset is not uniform:

| Problem | Where it happens |
| --- | --- |
| `2018-02-20` carries 4 extra identifier columns (`Flow ID`, `Src IP`, `Src Port`, `Dst IP`) | 1 of 10 files | 
| The CSV header is repeated *inside* the file as a data row | `2018-02-16` (1×), `2018-02-28` (33×), `2018-03-01` (25×) |
| Numbers are read as text whenever a column contains one of those header rows | the files above |

The section is built bottom-up, so each part can be read on its own:

| Part | Question it answers | Depends on |
| --- | --- | --- |
| 2.1.1 Settings | Where are the files, how big is a chunk? | — |
| 2.1.2 Column schema | Which columns does every output file have? | 2.1.1 |
| 2.1.3 Cleaning one chunk | How is a batch of rows made trustworthy? | 2.1.1, 2.1.2 |
| 2.1.4 Writing Parquet | What is a file called and how is it written? | 2.1.1 |
| 2.1.5 Converting one CSV | How is a single day converted? | 2.1.3, 2.1.4 |
| 2.1.6 Converting the folder | How are all 10 days run in parallel? | 2.1.2, 2.1.5 |
| 2.1.7 Run | — | 2.1.1, 2.1.6 |
| 2.1.8 Check the result | Did it write what we expect? | 2.1.1 |


### 2.1.1. Settings

Everything this step has to be *told*, kept in one place. The functions below take these as
arguments (with the constants as defaults), so any of them can also be called on a different
folder or chunk size without editing the function.

In [ ]:
PATH_FOLDER_CSV = "cse-cic-ids2018"
PATH_FOLDER_RAW = "data-raw"

In [ ]:
CHUNK_SIZE = 100_000

In [ ]:
LABEL_COLUMN = "label"
FEATURE_DTYPE = "float32"

In [ ]:
COLUMNS_TO_DROP = {
    "flow id",
    "src ip",
    "source ip",
    "src port",
    "source port",
    "dst ip",
    "destination ip",
    "timestamp",
}

### 2.1.2. Column Schema

*Which columns should every Parquet file have?*

Decided once, from the header rows alone — no data is read here. The schema is the union of the
column names of all CSVs, in first-seen order, minus `COLUMNS_TO_DROP`. Because `2018-02-20` is the
only file that differs and all four of its extra columns are dropped, the result is the same 79
columns for every day, which is what lets later steps read the whole folder as one table.

This is the **only** place `COLUMNS_TO_DROP` is used: one decision, in one place. 2.1.3 then forces
each chunk of data into the layout decided here.

In [ ]:
def normalize_column_names(column_names) -> list[str]:
    """Trim surrounding spaces and lowercase, so `" Dst Port"` and `"dst port"` are one name."""
    columns = []
    for column in column_names:
        columns.append(str(column).strip().lower())
    return columns

In [ ]:
def exclude_unwanted_columns(
    column_names: list[str], columns_to_drop: set[str] = COLUMNS_TO_DROP
) -> list[str]:
    """Keep only the column names worth storing, preserving their order."""
    remaining_columns = []
    for column in column_names:
        if column not in columns_to_drop:
            remaining_columns.append(column)
    return remaining_columns

In [ ]:
def read_csv_column_names(source_folder_name: str, file_name: str) -> list[str]:
    """Read the header row of one CSV — `nrows=0` loads no data — and normalize its names."""
    file_path = os.path.join(source_folder_name, file_name)
    header_only = pd.read_csv(file_path, nrows=0)
    return normalize_column_names(header_only.columns)

In [ ]:
def create_column_schema(
    source_folder_name: str, columns_to_drop: set[str] = COLUMNS_TO_DROP
) -> list[str]:
    """Build the column layout shared by every Parquet file written from this folder."""
    column_schema: list[str] = []
    for file_name in get_all_file_names_in_folder(source_folder_name, "csv"):
        file_columns = read_csv_column_names(source_folder_name, file_name)
        for column_name in exclude_unwanted_columns(file_columns, columns_to_drop):
            if column_name not in column_schema:
                column_schema.append(column_name)
    return column_schema

In [ ]:
column_schema = create_column_schema(PATH_FOLDER_CSV)
print(f"{len(column_schema)} columns: {column_schema[:3]} ... {column_schema[-2:]}")

### 2.1.3. Cleaning One Chunk

*How is a batch of raw rows made trustworthy?*

Four small transformations, each doing one thing, and `clean_chunk` running them in order. They all
take a `DataFrame` and return one, so the order is visible at a glance and any single step can be
tried on its own chunk while reading.

1. `standardize_column_names` — same naming rule as the schema, so the two can be matched.
2. `align_to_column_schema` — same columns, same order, in every file. This is also what removes
   the dropped identifier columns: they are simply not in the schema.
3. `cast_features_to_numeric` — text to float; anything unparseable becomes `NaN`.
4. `drop_invalid_label_rows` — throws away repeated header rows and rows without a label.

Order matters: the columns must be renamed before they can be matched against the schema, and the
label must still be text when the repeated header rows are detected — which is why the label is the
one column step 3 leaves alone.

In [ ]:
def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Rename the columns of a chunk with the same rule `create_column_schema` used."""
    df.columns = normalize_column_names(df.columns)
    return df

In [ ]:
def align_to_column_schema(df: pd.DataFrame, column_schema: list[str]) -> pd.DataFrame:
    """Force a chunk into the shared layout: schema columns only, in schema order.

    Columns the schema does not list (the dropped identifiers) disappear, and columns this file
    does not have would come back filled with `NaN`, so every output file has the same shape.
    """
    return df.reindex(columns=column_schema)

In [ ]:
def cast_features_to_numeric(
    df: pd.DataFrame, label_column: str = LABEL_COLUMN, dtype: str = FEATURE_DTYPE
) -> pd.DataFrame:
    """Convert every column except the label to a float.

    A column that contains a repeated header row is read as text by pandas, which would make the
    whole column text. `errors="coerce"` turns those values — and any blank — into `NaN`; the rows
    they came from are dropped next, and genuinely missing values are filled in 2.5 (Imputation).
    """
    feature_columns = [name for name in df.columns if name != label_column]
    df[feature_columns] = df[feature_columns].apply(pd.to_numeric, errors="coerce").astype(dtype)
    return df

In [ ]:
def drop_invalid_label_rows(df: pd.DataFrame, label_column: str = LABEL_COLUMN) -> pd.DataFrame:
    """Remove rows that are not observations.

    Some files repeat their CSV header in the middle of the data (2018-02-28 does it 33 times),
    which pandas reads as an ordinary row whose label is the literal text `"Label"`. Those rows go,
    together with any row that has no label at all.
    """
    label_values = df[label_column].astype("string").str.strip().str.lower()
    is_repeated_header = label_values == label_column
    return df[~is_repeated_header].dropna(subset=[label_column])

In [ ]:
def clean_chunk(df: pd.DataFrame, column_schema: list[str]) -> pd.DataFrame:
    """Run one chunk of raw CSV rows through the four steps above, in order."""
    df = standardize_column_names(df)
    df = align_to_column_schema(df, column_schema)
    df = cast_features_to_numeric(df)
    df = drop_invalid_label_rows(df)
    return df

### 2.1.4. Writing Parquet Files

*What is an output file called, and how is it written?*

One chunk becomes one file. Chunk numbers are zero-padded so the files of a day sort in the order
they were read, and the source CSV name is kept as the prefix — the train/validation/test split in
2.4 selects files by the date in that prefix, so the naming is not cosmetic.

`snappy` compression is the fast-to-decompress default; these files are read many times in the
steps that follow, so read speed matters more than the last few percent of disk space.

In [ ]:
def build_parquet_file_name(csv_file_name: str, chunk_number: int) -> str:
    """`2018-02-14-Wednesday_....csv` + chunk 7 -> `2018-02-14-Wednesday_..._00007.parquet`."""
    base_name = os.path.splitext(csv_file_name)[0]
    return f"{base_name}_{chunk_number:05d}.parquet"

In [ ]:
def write_dataframe_to_parquet(
    df: pd.DataFrame, target_folder_name: str, file_name: str
) -> str:
    """Write one cleaned chunk and return the path it was written to."""
    output_path = os.path.join(target_folder_name, file_name)
    df.to_parquet(output_path, engine="pyarrow", compression="snappy", index=False)
    return output_path

### 2.1.5. Converting One CSV File

*How is a single day converted?*

The two functions that turn 2.1.3 and 2.1.4 into a stream: read a fixed number of rows, clean them,
write them, forget them. Peak memory is one chunk, not one file, so the 4 GB `2018-02-20` costs no
more than the 108 MB `2018-03-01`.

`low_memory=False` makes pandas look at the whole column before choosing its type, instead of
guessing per block and reporting mixed types for the columns that contain a repeated header row.

In [ ]:
def read_csv_in_chunks(source_folder_name: str, file_name: str, chunk_size: int = CHUNK_SIZE):
    """Iterate over one CSV `chunk_size` rows at a time."""
    file_path = os.path.join(source_folder_name, file_name)
    return pd.read_csv(file_path, chunksize=chunk_size, low_memory=False)

In [ ]:
def convert_csv_file_to_parquet(
    source_folder_name: str,
    target_folder_name: str,
    file_name: str,
    column_schema: list[str],
    chunk_size: int = CHUNK_SIZE,
) -> int:
    """Convert one CSV into a numbered series of Parquet files; return how many were written."""
    written_files = 0
    for chunk_number, chunk in enumerate(
        read_csv_in_chunks(source_folder_name, file_name, chunk_size), start=1
    ):
        cleaned_chunk = clean_chunk(chunk, column_schema)
        write_dataframe_to_parquet(
            cleaned_chunk,
            target_folder_name,
            build_parquet_file_name(file_name, chunk_number),
        )
        written_files += 1
    return written_files

### 2.1.6. Converting Every CSV File

*How is the whole folder run?*

The only part that knows about parallelism. Each worker process takes one whole CSV, so the workers
never write to the same file and no result has to be sent back except a count.

The schema is built **once, before** the workers start, and passed in — if every worker derived its
own, a file would be aligned to a layout the others do not share.

`inner_max_num_threads=1` stops pandas and pyarrow from starting their own thread pools inside each
worker; without it, 10 processes × N threads would fight over the same cores and run slower.

In [ ]:
def convert_all_csv_files_to_parquet(
    source_folder_name: str,
    target_folder_name: str,
    chunk_size: int = CHUNK_SIZE,
) -> None:
    """Convert every CSV in the source folder to Parquet, one worker process per file."""
    csv_file_names = get_all_file_names_in_folder(source_folder_name, "csv")
    column_schema = create_column_schema(source_folder_name)
    os.makedirs(target_folder_name, exist_ok=True)

    with parallel_config(backend="loky", inner_max_num_threads=1):
        written_per_file = Parallel(n_jobs=-1, verbose=5)(
            delayed(convert_csv_file_to_parquet)(
                source_folder_name,
                target_folder_name,
                file_name,
                column_schema,
                chunk_size,
            )
            for file_name in csv_file_names
        )

    print(
        f"Converted {len(csv_file_names)} CSV files "
        f"into {sum(written_per_file)} Parquet files in {target_folder_name}/."
    )

### 2.1.7. Run the Conversion

Reads `cse-cic-ids2018/` and fills `data-raw/`. Roughly 6.7 GB in, and it is the slowest step in the
notebook — everything after this reads Parquet, so it only has to be run once.

In [ ]:
convert_all_csv_files_to_parquet(PATH_FOLDER_CSV, PATH_FOLDER_RAW)

### 2.1.8. Check the Result

A quick look at what landed on disk: one row per source CSV, how many Parquet files it produced and
how many rows survived cleaning. Row counts come from the Parquet footers, so nothing is loaded.

In [ ]:
def summarize_parquet_folder(target_folder_name: str) -> pd.DataFrame:
    """Count the files and rows written per source CSV."""
    counts_per_source: dict[str, dict[str, int]] = {}
    for file_name in get_all_file_names_in_folder(target_folder_name, "parquet"):
        source_name = file_name.rsplit("_", 1)[0]
        row_count = (
            pl.scan_parquet(os.path.join(target_folder_name, file_name))
            .select(pl.len())
            .collect()
            .item()
        )
        counts = counts_per_source.setdefault(source_name, {"files": 0, "rows": 0})
        counts["files"] += 1
        counts["rows"] += row_count
    return pd.DataFrame(
        [{"source": name, **counts} for name, counts in sorted(counts_per_source.items())]
    ).set_index("source")

In [ ]:
summarize_parquet_folder(PATH_FOLDER_RAW)